# QuantJourney SDK - FINRA Short Interest

This notebook demonstrates FINRA data:
- Short interest reports
- Short squeeze analysis
- Market sentiment indicators

**API:** https://api.quantjourney.cloud

In [ ]:
import sys
sys.path.insert(0, '..')
from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'png'
import os
API_KEY = os.environ['QJ_API_KEY']
qj = QuantJourneyAPI(api_key=API_KEY)
print('OK Connected to QuantJourney API')


## 1. Short Interest - Tesla

In [ ]:
try:
    response = qj.finra.get_short_interest(symbol='TSLA')
    short_data = response.get('value', response) if isinstance(response, dict) else response
    if short_data:
        df_short = pd.DataFrame(short_data) if isinstance(short_data, list) else pd.DataFrame([short_data])
        print(f'TSLA Short Interest Records: {len(df_short)}')
        print(f'\nAvailable columns:')
        for col in df_short.columns:
            print(f'  - {col}')
except Exception as e:
    print(f'Note: FINRA short interest - {e}')


In [ ]:
if 'df_short' in dir() and len(df_short) > 0:
    date_col = None
    for col in ['settlementDate', 'date', 'reportDate']:
        if col in df_short.columns:
            date_col = col
            break
    if date_col:
        df_short['date'] = pd.to_datetime(df_short[date_col])
        df_short = df_short.sort_values('date')
    value_col = None
    for col in ['shortInterest', 'short_interest', 'currentShortPositionQuantity', 'shortPosition']:
        if col in df_short.columns:
            value_col = col
            break
    if value_col and date_col:
        fig = go.Figure()
        fig.add_trace(go.Bar(x=df_short['date'], y=df_short[value_col], name='Short Interest', marker_color='red'))
        fig.update_layout(title='TSLA Short Interest Over Time', yaxis_title='Shares Short', xaxis_title='Settlement Date', template='plotly_dark', height=450)
        fig.show()
        print(f'\nTSLA Short Interest Analysis:')
        print(f'  Latest:  {df_short[value_col].iloc[-1]:,.0f} shares')
        print(f'  Average: {df_short[value_col].mean():,.0f} shares')
        print(f'  Max:     {df_short[value_col].max():,.0f} shares')
        print(f'  Min:     {df_short[value_col].min():,.0f} shares')
    else:
        print('\nLatest data:')
        print(df_short.tail())


## 2. Short Interest - Multiple Stocks

In [ ]:
symbols = ['TSLA', 'GME', 'AMC', 'AAPL', 'NVDA']
short_data_all = {}
print('Short Interest Comparison')
print('=' * 60)
for symbol in symbols:
    try:
        response = qj.finra.get_short_interest(symbol=symbol)
        data = response.get('value', response) if isinstance(response, dict) else response
        if data:
            df = pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame([data])
            value_col = None
            for col in ['shortInterest', 'short_interest', 'currentShortPositionQuantity']:
                if col in df.columns:
                    value_col = col
                    break
            if value_col:
                latest = df[value_col].iloc[-1]
                short_data_all[symbol] = latest
                print(f'{symbol:>6}: {latest:>15,.0f} shares short')
    except Exception as e:
        print(f'{symbol:>6}: Error - {e}')


In [ ]:
if short_data_all:
    fig = go.Figure(data=[go.Bar(x=list(short_data_all.keys()), y=list(short_data_all.values()), marker_color=['red' if v > 10000000 else 'orange' for v in short_data_all.values()])])
    fig.update_layout(title='Short Interest Comparison', yaxis_title='Shares Short', xaxis_title='Symbol', template='plotly_dark', height=400)
    fig.show()


## 3. Short Squeeze Indicators

In [ ]:
print('=' * 60)
print('SHORT SQUEEZE ANALYSIS GUIDE')
print('=' * 60)
print('\nKEY METRICS:')
metrics = [('Short Interest', 'Total shares sold short'), ('Short % of Float', 'Short interest / freely traded shares'), ('Days to Cover', 'Short interest / avg daily volume'), ('Short Ratio', 'Short interest vs total shares outstanding'), ('Cost to Borrow', 'Fee rate to borrow shares for shorting')]
for metric, desc in metrics:
    print(f'   • {metric:<20} - {desc}')
print('\nSHORT SQUEEZE WARNING SIGNS:')
signs = ['Short % of float > 20%', 'Days to cover > 5', 'Rising short interest + rising price', 'High cost to borrow (>50% annualized)', 'Catalyst (earnings, news) approaching']
for sign in signs:
    print(f'   Warning  {sign}')
print('\n' + '=' * 60)


## Summary

FINRA short interest data covered:
- **Short Interest**: Shares sold short
- **Historical Trends**: Short interest over time
- **Cross-Stock Comparison**: Multiple symbols

### Trading Implications:
- High short interest = potential squeeze
- Rising shorts on rising price = bearish conviction
- Falling shorts on rising price = covering (bullish)